# TRABAJO PRÁCTICO: FILTRADO, IDENTIFICACIÓN DE SISTEMAS Y COHERENCIA CUADRÁTICA

**Integrantes:** Ferreyra Florencia, González Tomás, Molina Lara y Scafati Jerónimo  
**Fecha:** 03/07/2026  
**Cátedra:** Procesamiento Digital de Señales (DSP)  

---


## Introducción

El objetivo del presente trabajo consiste en el desarrollo y análisis de herramientas para la caracterización de filtros y la identificación de sistemas lineales e invariantes en el tiempo (LTI). Para ello, se diseñaron y evaluaron filtros de media móvil, peine y FIR con ventana Hamming. Las funciones de procesamiento y visualización se centralizan en `functions.py` para evitar duplicación de código e importar desde la librería central `core/dsp`.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "faculty" / "preentrega"))

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from functions import (
    add_white_noise,
    apply_fir,
    compute_fft,
    compute_frequency_response,
    convolve_frequency,
    convolve_time,
    filtro_fir,
    filtro_media_movil,
    filtro_peine,
    generate_impulse,
    generate_pure_tones,
    load_audio,
    load_fir_coefficients,
    plot_frequency_response,
    truncate_fir,
)

Asimismo, la segunda parte del Trabajo Práctico se enfoca en la **caracterización e identificación de sistemas desconocidos** a partir de los pares de señales de entrada $x[n]$ y salida $y[n]$ provistas por la cátedra en la carpeta `archivos/`.

Mediante el estimador no paramétrico de respuesta en frecuencia $H_1(\omega)$ y la **coherencia cuadrática** $\gamma_{xy}^2(\omega)$ basados en el **método de Welch** (promediado de densidades espectrales de potencia $G_{xx}$, $G_{yy}$ y espectro cruzado $G_{xy}$), se analiza el comportamiento frecuencial del sistema, su linealidad por bandas y el efecto de la relación señal a ruido (SNR).

## 1. Respuesta al Impulso y Caracterización Temporal

La **respuesta al impulso** $h[n]$ permite describir las modificaciones que un determinado sistema LTI le aplica a una señal. De esta manera, los sistemas LTI se pueden caracterizar por completo mediante la convolución lineal:

$$y[n] = x[n] * h[n] = \sum_{k=-\infty}^{\infty} x[k]\, h[n-k]$$

Para analizar el comportamiento de distintos filtros, se realizaron tres tipos de filtros digitales: **media móvil**, **peine** y **FIR**. En particular, para el filtro de media móvil, se proponen configuraciones para una, dos o tres pasadas. Luego, se generó una señal impulso discreta y se la utilizó como entrada para los filtros para obtener la respuesta al impulso de cada uno de ellos.


In [ ]:
# 1. Generamos un impulso discreto de longitud N = 2048 para abarcar completamente la respuesta FIR (L = 1457)
length = 2048
impulse = generate_impulse(length, delay=0)

# 2. Pasamos el impulso por los distintos filtros para extraer h[n]
h_ma, _ = filtro_media_movil(impulse, m=8, p=1)
h_comb, _ = filtro_peine(impulse, a=0.5, b=0.3, c=0.2)
h_fir, h_coefs = filtro_fir(impulse)

# 3. Graficamos las respuestas al impulso completas
plt.figure(figsize=(12, 9))

# Media Móvil (Zoom en las primeras 30 muestras para apreciar M=8)
plt.subplot(3, 1, 1)
t_sub = np.arange(30)
plt.stem(t_sub, h_ma[:30], basefmt=" ")
plt.title("Respuesta al Impulso: Media Móvil (M=8, passes=1)")
plt.xlabel("Muestras [n]")
plt.ylabel("Amplitud")
plt.grid(True)

# Filtro Peine (Zoom en las primeras 10 muestras para apreciar coeficientes)
plt.subplot(3, 1, 2)
t_comb = np.arange(10)
plt.stem(t_comb, h_comb[:10], basefmt=" ")
plt.title("Respuesta al Impulso: Filtro Peine (b0=0.5, b1=0.3, b2=0.2)")
plt.xlabel("Muestras [n]")
plt.ylabel("Amplitud")
plt.grid(True)

# Filtro FIR Completo (Mostrando todas las L = 1457 muestras del filtro)
plt.subplot(3, 1, 3)
t_fir = np.arange(len(h_coefs))
plt.plot(t_fir, h_coefs, color='green', linewidth=1.5)
plt.title(f"Respuesta al Impulso Completa: Filtro FIR de la Cátedra (L = {len(h_coefs)} coeficientes, Ventana Hamming)")
plt.xlabel("Muestras [n]")
plt.ylabel("Amplitud")
plt.grid(True)

plt.tight_layout()
plt.show()


### Análisis y Conclusiones de la Sección 1:

En los gráficos es posible observar las características temporales de cada filtro. El de media móvil es un pulso rectangular de amplitud $1/M = 0.125$ y duración $M=8$, consistente con el promedio aritmético de las últimas $M$ muestras. El filtro peine exhibe exactamente 3 muestras no nulas, correspondientes a los coeficientes $b_0=0.5$, $b_1=0.3$ y $b_2=0.2$ en los instantes $n=0, 1, 2$. Se trata entonces de un filtro de respuesta al impulso finita de segundo orden. Por último, el filtro FIR presenta una mayor duración ($L=1457$ coeficientes) con forma de sinc, suavizada por la ventana Hamming. La simetría alrededor del pico central ($n=728$) confirma la fase lineal. 

## 2. Caracterización en Frecuencia

A partir de la transformada de Fourier de las respuestas al impulso, se obtuvo la respuesta en frecuencia de cada sistema, incluyendo tanto magnitud como fase:

$$H(\omega) = \frac{Y(\omega)}{X(\omega)}$$

Al excitar con un impulso discreto completo ($X(\omega)=1$), la respuesta en frecuencia coincide con la transformada de Fourier de la respuesta al impulso completa $h[n]$:

$$H(\omega) = \mathcal{F}\{h[n]\}$$

Se grafican la magnitud (en dB, acotando el eje Y a $[-60, 5]\text{ dB}$ para evitar distorsiones por ceros numéricos) y la fase (en radianes desenrollada) de los tres filtros.


In [ ]:
fs = 44100
# Impulso largo de N = 2048 para capturar la respuesta al impulso FIR completa sin truncar la cola
imp_cal = generate_impulse(2048, delay=0)

# Obtener las respuestas al impulso causales completas desde el origen
h_ma_c, _ = filtro_media_movil(imp_cal, m=8, p=1)
h_comb_c, _ = filtro_peine(imp_cal, a=0.5, b=0.3, c=0.2)
h_fir_c, _ = filtro_fir(imp_cal)

# Calcular H(w) sobre la señal completa
freqs_ma, H_ma = compute_frequency_response(imp_cal, h_ma_c, fs)
freqs_comb, H_comb = compute_frequency_response(imp_cal, h_comb_c, fs)
freqs_fir, H_fir = compute_frequency_response(imp_cal, h_fir_c, fs)

# Mostrar gráficos de respuesta en frecuencia acotando la magnitud a [-60, 5] dB
fig_ma = plot_frequency_response(freqs_ma, H_ma, title="Respuesta en Frecuencia: Media Móvil (M=8)", ylim=(-60, 5))
plt.show()

fig_comb = plot_frequency_response(freqs_comb, H_comb, title="Respuesta en Frecuencia: Filtro Peine", ylim=(-60, 5))
plt.show()

fig_fir = plot_frequency_response(freqs_fir, H_fir, title="Respuesta en Frecuencia Completa: Filtro FIR (Cátedra)", ylim=(-60, 5))
plt.show()


### Análisis y Conclusiones de la Sección 2:
- **Filtro de Media Móvil**: Presenta un comportamiento pasabajos de baja selectividad, con una atenuación progresiva a medida que aumenta la frecuencia. Además, se observan nulos periódicos en múltiplos de $f_s/M$ ($5512.5\text{ Hz}$), coincidentes con los saltos de $\pi$ en la fase.
- **Filtro Peine**: Exhibe una respuesta frecuencial periódica. Con coeficientes positivos, actúa como pasabajos suave con atenuación moderada y fase continua.
- **Filtro FIR de la Cátedra**: Presenta una excelente selectividad pasabajos con una frecuencia de corte nítida en $\approx 1000\text{ Hz}$ y alta atenuación en la banda de parada ($>40\text{ dB}$). En la banda de paso, la fase es perfectamente lineal, asegurando retardo de grupo constante sin distorsión de fase.


## 3. Variación de Parámetros

Luego, se analizó el efecto de cambiar los parámetros clave del filtro de media móvil y peine:
1. **Media Móvil**: tamaño de ventana $M \in \{3, 8, 20\}$.
2. **Filtro Peine**: variación de coeficientes $b_0, b_1, b_2$.


In [ ]:
fs = 44100
imp_cal = generate_impulse(2048, delay=0)

# 1. Variación de M en Media Móvil
plt.figure(figsize=(10, 5))
for M in [3, 8, 20]:
    h_sweep, _ = filtro_media_movil(imp_cal, m=M, p=1)
    freqs, H = compute_frequency_response(imp_cal, h_sweep, fs=fs)
    mag_db = 20 * np.log10(np.clip(np.abs(H), 1e-15, None))
    plt.plot(freqs, mag_db, label=f"M = {M}")

plt.xscale('log')
plt.title("Media Móvil: Variación del tamaño de ventana M")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud [dB]")
plt.ylim(-60, 5)
plt.grid(True, which='both')
plt.legend()
plt.show()

# 2. Variación de coeficientes en Filtro Peine
plt.figure(figsize=(10, 5))
casos_peine = [
    {"a": 1.0, "b": 0.0, "c": 0.0, "lbl": "Identidad (a=1)"},
    {"a": 0.5, "b": 0.5, "c": 0.0, "lbl": "Suma (a=0.5, b=0.5) -> Pasabajos (nulo en fs/2)"},
    {"a": 0.5, "b": -0.5, "c": 0.0, "lbl": "Resta (a=0.5, b=-0.5) -> Pasaaltos (nulo en DC)"},
    {"a": 0.5, "b": 0.0, "c": -0.5, "lbl": "Interferencia (a=0.5, c=-0.5) -> Nulo en fs/4"}
]

for caso in casos_peine:
    h_sweep, _ = filtro_peine(imp_cal, a=caso["a"], b=caso["b"], c=caso["c"])
    freqs, H = compute_frequency_response(imp_cal, h_sweep, fs=fs)
    mag_db = 20 * np.log10(np.clip(np.abs(H), 1e-15, None))
    plt.plot(freqs, mag_db, label=caso["lbl"])

plt.xscale('log')
plt.title("Filtro Peine: Variación de Coeficientes")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud [dB]")
plt.ylim(-60, 5)
plt.grid(True, which='both')
plt.legend()
plt.show()


### Análisis y Conclusiones de la Sección 3:

Para el filtro de media móvil, se obtuvo que, al aumentar el largo de la ventana $M$, la respuesta al impulso se extiende en el tiempo, ya que hay una mayor cantidad de muestras. En consecuencia, la respuesta en frecuencia es más selectiva, el ancho de la banda de paso disminuye y los nulos espectrales se acercan entre sí. Por otro lado, ocurre una mayor atenuación de las componentes de frecuencia media y alta, por lo que la señal filtrada presenta un mayor suavizado en el dominio temporal. 
Con respecto al filtro peine, el comportamiento depende de los valores de los coeficientes y su posición temporal. Cuando existe un único coeficiente no nulo e igual a 1 ($b_0=1$), se obtiene una respuesta plana de 0 dB, que se corresponde con un sistema identidad, es decir, no hay una diferencia entre la entrada y la salida. Cuando los primeros dos coeficientes son iguales y positivos ($b_0=0.5, b_1=0.5$), el comportamiento es pasa bajos, por el contrario, al ser iguales en módulo pero de signo opuesto ($b_0=0.5, b_1=-0.5$), se convierte en un pasa altos. Por último, si el primer y tercer coeficiente son opuestos ($b_0=0.5, b_2=-0.5$), se encuentran nulos periódicos en un cuarto de la frecuencia de muestreo ($f_s/4$).



## 4. Señales de Prueba

Se generaron dos señales para evaluar los filtros:
1. **Mezcla de Tonos Ruidosa**: suma de tres senoidales ($500\text{ Hz}$, $1000\text{ Hz}$ y $5000\text{ Hz}$) con amplitudes $1.0$, $0.5$ y $0.2$, contaminada con ruido blanco gaussiano a $\text{SNR} = 15\text{ dB}$.
2. **Audio Musical Ruidoso**: señal provista por la cátedra, contaminada con ruido blanco a $\text{SNR} = 15\text{ dB}$.

In [ ]:
fs = 44100
duracion = 1.0

# 1. Generación de mezcla de tonos limpios y adición de ruido SNR = 15 dB
tonos_limpios = generate_pure_tones(frequencies=[500.0, 1000.0, 5000.0], amplitudes=[1.0, 0.5, 0.2], fs=fs, duration=duracion)
tonos_ruidosos = add_white_noise(tonos_limpios, snr_db=15.0)

# 2. Carga de audio musical de la cátedra y adición de ruido SNR = 15 dB
# Usamos un bucle de búsqueda hacia arriba para encontrar el directorio raíz del proyecto
path = os.getcwd()
while path != '/' and not os.path.exists(os.path.join(path, "archivos")):
    path = os.path.dirname(path)
project_root = path

wav_path = os.path.join(project_root, "archivos", "musica_ruido_0.05.wav") # Archivo base de prueba
musica_limpia, fs_music = load_audio(wav_path)
if len(musica_limpia.shape) > 1:
    musica_limpia = musica_limpia[:, 0]  # Mono
musica_ruidosa = add_white_noise(musica_limpia, snr_db=15.0)

# 3. Visualización temporal (primeros ms para ver la forma de onda)
t_tonos = np.arange(len(tonos_limpios)) / fs
slice_tonos = slice(0, 400)

plt.figure(figsize=(12, 5))
plt.plot(t_tonos[slice_tonos], tonos_limpios[slice_tonos], label="Tonos Limpios", alpha=0.7)
plt.plot(t_tonos[slice_tonos], tonos_ruidosos[slice_tonos], label="Tonos Ruidosos (SNR=15 dB)", alpha=0.9, color='orange')
plt.title("Visualización Temporal: Mezcla de Tonos")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud")
plt.legend()
plt.grid(True)
plt.show()

# 4. Visualización espectral (espectros de magnitud de Fourier)
freqs_t, mags_t_limpio = compute_fft(tonos_limpios, fs)
_, mags_t_ruido = compute_fft(tonos_ruidosos, fs)

plt.figure(figsize=(12, 5))
plt.plot(freqs_t, mags_t_limpio, label="Espectro Limpio", alpha=0.7)
plt.plot(freqs_t, mags_t_ruido, label="Espectro Ruidoso", alpha=0.5, color='orange')
plt.title("Visualización Espectral: Espectro de Amplitud de Tonos")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud")
plt.xlim(0, 6000)
plt.legend()
plt.grid(True)
plt.show()

### Análisis y Conclusiones de la Sección 4:

En el tiempo, se observa que el ruido blanco superpone fluctuaciones aleatorias de alta frecuencia que distorsionan la forma de onda original. Por otro lado, en frecuencia, el espectro de la señal limpia muestra 3 picos bien definidos en $500$, $1000$ y $5000\text{ Hz}$. El ruido blanco eleva el piso espectral de forma uniforme en todo el ancho de banda, pero los tonos permanecen identificables dado su alta potencia relativa.

## 5. Filtrado en Tiempo y Frecuencia

El filtrado FIR puede realizarse por dos vías equivalentes:

1. **Convolución lineal en el tiempo:**
   $$y[n] = x[n] * h[n] = \sum_{k=0}^{M-1} h[k]\, x[n-k]$$

2. **Multiplicación espectral (Teorema de la Convolución):**
   $$y[n] = \mathcal{F}^{-1}\{X(\omega) \cdot H(\omega)\}$$

La multiplicación de DFT/FFT implementa internamente una **convolución circular**. Para que sea idéntica a la convolución lineal, se aplica **zero-padding** a ambas señales hasta una longitud mínima:

$$N_{\text{fft}} \ge N_x + N_h - 1$$

Sin este relleno, ocurre *aliasing* en el dominio del tiempo (solapamiento circular de la cola del filtro sobre el inicio de la señal).

A continuación se filtra la señal de tonos ruidosa por ambos métodos y se mide la discrepancia numérica entre ellos.

In [ ]:
# Cargar los coeficientes FIR provistos por la cátedra para el filtrado
coefs_path = os.path.join(project_root, "archivos", "fir_hamming_1000Hz.npy")
h_fir = load_fir_coefficients(coefs_path)

# 1. Filtrado en tiempo usando convolución lineal completa
y_time = convolve_time(tonos_ruidosos, h_fir)

# 2. Filtrado en frecuencia usando convolución circular con padding N_fft
y_freq = convolve_frequency(tonos_ruidosos, h_fir)

# 3. Calcular la norma de la diferencia (discrepancia numérica)
error_norm = np.linalg.norm(y_time - y_freq)
print(f"Norma de la diferencia entre ambos métodos: {error_norm:.2e}")
print(f"¿Son numéricamente equivalentes?: {np.allclose(y_time, y_freq)}")

### Análisis Comparativo: Convolución Temporal vs. Multiplicación Espectral

La diferencia entre ambos métodos ($y_{\text{tiempo}}[n]$ vs. $\mathcal{F}^{-1}\{X(\omega)\cdot H(\omega)\}$) es del orden de $10^{-15}$, atribuible a errores de redondeo en aritmética de punto flotante. Ambos métodos son, a efectos prácticos, idénticos.

Esta equivalencia es exacta únicamente cuando se respeta la condición de zero-padding $N \ge N_x + N_h - 1$. De lo contrario, la periodicidad implícita de la DFT introduce **aliasing circular** que corrompe el resultado.

## 6. Truncado de Coeficientes del Filtro FIR y Fenómeno de Gibbs

En la práctica, la respuesta al impulso de un filtro FIR de orden elevado ($L=1457$) puede truncarse a $N < L$ coeficientes para reducir el costo computacional y el retardo. Para analizar correctamente el efecto del truncado manteniendo la **fase lineal**, el recorte debe realizarse de forma **simétrica** alrededor del pico central ($n=728$).

El truncado simétrico equivale a multiplicar $h[n]$ por una ventana rectangular centrada $w_r[n]$, lo que en frecuencia conlleva una convolución con una función sinc (kernel de Dirichlet):

$$h_{\text{trunc}}[n] = h[n] \cdot w_r[n] \quad\longrightarrow\quad H_{\text{trunc}}(\omega) = \frac{1}{2\pi}\int_{-\pi}^{\pi} H(\theta)\,W_r(\omega-\theta)\,d\theta$$

Esto produce dos efectos principales:
1. **Ensanchamiento de la banda de transición**: La pendiente de corte disminuye al reducir $N$.
2. **Fenómeno de Gibbs**: Rizado (*ripple*) en las bandas de paso y de parada debido a los lóbulos secundarios de la ventana rectangular.

A continuación se evalúa el truncado simétrico para $N \in \{51, 151, 301, 601, 1001\}$ coeficientes frente al filtro completo de $L=1457$.


In [ ]:
coefs_path = os.path.join(project_root, "archivos", "fir_hamming_1000Hz.npy")
h_original = load_fir_coefficients(coefs_path)
L_orig = len(h_original)
fs = 44100

# Impulso largo para caracterizar el espectro con resolución adecuada (N = 2048)
imp_eval = generate_impulse(2048, delay=0)

plt.figure(figsize=(12, 6))

# 1. Respuesta en frecuencia del original (L = 1457)
y_orig = apply_fir(imp_eval, h_original)
freqs_eval, H_orig = compute_frequency_response(imp_eval, y_orig, fs=fs)
mag_orig = 20 * np.log10(np.clip(np.abs(H_orig), 1e-15, None))
plt.plot(freqs_eval, mag_orig, label=f"Original Completo (L = {L_orig} coefs)", linewidth=2.5, color='black')

# 2. Respuestas para truncados simétricos N = 51, 151, 301, 601, 1001
colores = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']
for i, N in enumerate([51, 151, 301, 601, 1001]):
    h_trunc = truncate_fir(h_original, N, mode='symmetric')
    y_trunc = apply_fir(imp_eval, h_trunc)
    _, H_trunc = compute_frequency_response(imp_eval, y_trunc, fs=fs)
    mag_trunc = 20 * np.log10(np.clip(np.abs(H_trunc), 1e-15, None))
    plt.plot(freqs_eval, mag_trunc, label=f"Truncado Simétrico N = {N}", color=colores[i], alpha=0.8, linewidth=1.5)

plt.xscale('log')
plt.title("Efecto del Truncado Simétrico de Coeficientes en el Filtro FIR")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud [dB]")
plt.ylim(-60, 5)
plt.grid(True, which='both')
plt.legend()
plt.show()


### Evaluación del Truncado Simétrico y el Fenómeno de Gibbs:

Al recortar el filtro FIR de $L=1457$ coeficientes de forma **simétrica** alrededor de su centro ($n=728$):

1. **Fase Lineal Preservada**: El recorte simétrico garantiza que la simetría par del filtro se conserve, manteniendo la fase estrictamente lineal y evitando distorsiones de fase en la señal procesada.
2. **Efecto de la Longitud $N$**:
   - **$N=51$ y $N=151$**: Al descartar una porción significativa de los lóbulos secundarios en el tiempo, la ventana rectangular en frecuencia provoca una banda de transición ancha y disminuye la capacidad de rechazo en la banda de parada.
   - **$N=301$ y $N=601$**: La pendiente de corte se vuelve apreciablemente más pronunciada y la respuesta se aproxima de forma progresiva al filtro original.
   - **$N=1001$**: La respuesta en frecuencia es casi indistinguible del filtro completo original ($L=1457$).
3. **Fenómeno de Gibbs**: En las versiones truncadas se observa el oscilamiento o *ripple* característico en la banda de paso y de parada adyacente a la frecuencia de corte ($1000\text{ Hz}$). Con el truncado simétrico, el incremento de $N$ concentra los oscilamientos más cerca de la discontinuidad frecuencial sin alterar el carácter de fase lineal del sistema.


## 7. Demostración Matemática de la Coherencia (Teorema de Wiener-Khinchin)

A continuación se demuestra por qué la coherencia cuadrática vale **1** para un sistema LTI ideal sin ruido no correlacionado, utilizando la definición de las densidades espectrales de potencia como transformadas de Fourier de las funciones de correlación cruzada y autocorrelación.

### 7.1 Relación en el Dominio del Tiempo
Dado un sistema Lineal e Invariante en el Tiempo (LTI) caracterizado por su respuesta al impulso $h[n]$, la salida $y[n]$ ante una entrada $x[n]$ es la convolución discreta:
$$y[n] = h[n] * x[n] = \sum_{k=-\infty}^{\infty} h[k] \, x[n-k]$$

### 7.2 Funciones de Correlación
- **Correlación cruzada entre entrada y salida $R_{xy}[m]$:**
  $$R_{xy}[m] = \sum_{n=-\infty}^{\infty} x[n] \, y[n+m] = h[m] * R_{xx}[m]$$
- **Autocorrelación de la salida $R_{yy}[m]$:**
  $$R_{yy}[m] = h[m] * h[-m] * R_{xx}[m]$$

### 7.3 Aplicación del Teorema de Wiener-Khinchin
Por el **Teorema de Wiener-Khinchin**, las densidades espectrales de potencia (PSD) corresponden a la Transformada de Fourier de la autocorrelación o correlación cruzada:
- **Densidad espectral de potencia de la entrada $G_{xx}(\omega)$:**
  $$G_{xx}(\omega) = \mathcal{F}\{R_{xx}[m]\}$$
- **Densidad espectral de potencia cruzada $G_{xy}(\omega)$:**
  $$G_{xy}(\omega) = \mathcal{F}\{R_{xy}[m]\} = \mathcal{F}\{h[m] * R_{xx}[m]\} = H(\omega) \, G_{xx}(\omega)$$
- **Densidad espectral de potencia de la salida $G_{yy}(\omega)$:**
  Teniendo en cuenta que $h[-m] \xrightarrow{\mathcal{F}} H^*(\omega)$ (complejo conjugado):
  $$G_{yy}(\omega) = \mathcal{F}\{R_{yy}[m]\} = H(\omega) \, H^*(\omega) \, G_{xx}(\omega) = |H(\omega)|^2 \, G_{xx}(\omega)$$

### 7.4 Sustitución en la Ecuación de Coherencia Cuadrática
La función de coherencia cuadrática se define como:
$$\gamma_{xy}^2(\omega) = \frac{|G_{xy}(\omega)|^2}{G_{xx}(\omega) \, G_{yy}(\omega)}$$

Sustituyendo $G_{xy}(\omega) = H(\omega) G_{xx}(\omega)$ y $G_{yy}(\omega) = |H(\omega)|^2 G_{xx}(\omega)$:
$$\gamma_{xy}^2(\omega) = \frac{|H(\omega) \, G_{xx}(\omega)|^2}{G_{xx}(\omega) \left[|H(\omega)|^2 \, G_{xx}(\omega)\right]} = \frac{|H(\omega)|^2 \, G_{xx}^2(\omega)}{|H(\omega)|^2 \, G_{xx}^2(\omega)} = 1$$

$$\bbox[10px,border:2px solid #83BCA9]{\gamma_{xy}^2(\omega) \equiv 1 \quad \forall \omega \text{ donde } |H(\omega)| > 0 \text{ y } G_{xx}(\omega) > 0}$$

### 7.5 Interpretación Física de los Valores de Coherencia
- **$\gamma_{xy}^2(\omega) = 1$**: Indica una relación lineal perfecta y pura entre entrada y salida en la frecuencia $\omega$. Todo el contenido frecuencial en la salida es causado exclusivamente por la entrada a través del sistema LTI.
- **$\gamma_{xy}^2(\omega) = 0$**: La salida en esa frecuencia no guarda relación lineal alguna con la entrada (ausencia de transmisión o presencia exclusiva de ruido de medición no correlacionado).
- **$0 < \gamma_{xy}^2(\omega) < 1$**: Ocurre en la práctica debido a:
  1. Presencia de **ruido no correlacionado** en la medición (baja relación señal a ruido SNR).
  2. **No linealidades** en el sistema (distorsión armónica, saturación de componentes).
  3. Fugas espectrales por truncado temporal / selección de ventana.

## 8. Carga y Procesamiento de Datos Provistos por la Cátedra

Acorde a la consigna oficial, se procesaron los archivos de audio provistos en la carpeta `archivos/`:

1. **Señales de Entrada Provistas ($x[n]$):**
   - **Tonos puros con ruido aditivo:** `tonos_ruido_0.01.wav`, `tonos_ruido_0.05.wav`, `tonos_ruido_0.20.wav`.
   - **Señal musical con ruido aditivo:** `musica_ruido_0.01.wav`, `musica_ruido_0.05.wav`, `musica_ruido_0.20.wav`.

2. **Sistema bajo prueba ($h[n]$):**
   - Coeficientes del sistema desconocido: `fir_hamming_1000Hz.npy`.

3. **Generación de la salida ($y[n]$):**
   - La respuesta del sistema $y[n] = h[n] * x[n] + n_y[n]$ incluye ruido de medición no correlacionado en el canal de salida para evaluar cuantitativamente la coherencia bajo distintas relaciones señal a ruido (SNR).

In [ ]:
# Configuración del path de búsqueda del proyecto

project_root = Path.cwd()
while not (project_root / "faculty").exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "faculty" / "final"))

from functions import (
    identificar_sistema,
    evaluar_coherencia,
    load_audio,
    load_fir_coefficients,
    apply_fir,
    add_white_noise,
    plot_frequency_response,
    plot_coherence,
)

print("Entorno y módulos académicos cargados correctamente.")

In [ ]:
archivos_path = project_root / "archivos"
h_sistema = load_fir_coefficients(str(archivos_path / "fir_hamming_1000Hz.npy"))

niveles_ruido = [0.01, 0.05, 0.20]
resultados_tonos = {}
resultados_musica = {}

np.random.seed(42)

for amp_ruido in niveles_ruido:
    # 1. Procesamiento de Tonos Puros
    fn_tonos = f"tonos_ruido_{amp_ruido:.2f}.wav"
    x_tonos, fs = load_audio(str(archivos_path / fn_tonos))
    y_tonos_puro = apply_fir(x_tonos, h_sistema)
    ruido_y = amp_ruido * np.random.normal(0, 1, len(x_tonos))
    y_tonos = y_tonos_puro + ruido_y
    
    freqs_t, H_tonos = identificar_sistema(x_tonos, y_tonos, fs=fs, window_size=1024)
    _, coh_tonos = evaluar_coherencia(x_tonos, y_tonos, fs=fs, window_size=1024)
    resultados_tonos[amp_ruido] = (freqs_t, H_tonos, coh_tonos)

    # 2. Procesamiento de Señal Musical
    fn_musica = f"musica_ruido_{amp_ruido:.2f}.wav"
    x_musica, fs = load_audio(str(archivos_path / fn_musica))
    y_musica_puro = apply_fir(x_musica, h_sistema)
    ruido_y_m = amp_ruido * np.random.normal(0, 1, len(x_musica))
    y_musica = y_musica_puro + ruido_y_m
    
    freqs_m, H_musica = identificar_sistema(x_musica, y_musica, fs=fs, window_size=1024)
    _, coh_musica = evaluar_coherencia(x_musica, y_musica, fs=fs, window_size=1024)
    resultados_musica[amp_ruido] = (freqs_m, H_musica, coh_musica)

print("Identificación de sistemas y cálculo de coherencia completados exitosamente.")

## 9. Gráficos Comparativos de Respuesta en Frecuencia y Coherencia Cuadrática, y Análisis de Resultados

A continuación se presentan los gráficos de respuesta en frecuencia y coherencia cuadrática para cada par de señales de entrada y salida.

In [ ]:
# Gráfico 1: Identificación y Coherencia para Entrada de Tonos Puros
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

for amp_ruido in niveles_ruido:
    freqs, H, coh = resultados_tonos[amp_ruido]
    mag_db = 20 * np.log10(np.clip(np.abs(H), 1e-12, None))
    ax1.semilogx(freqs, mag_db, label=f"Nivel Ruido = {amp_ruido}", linewidth=1.5)
    ax2.semilogx(freqs, coh, label=f"Nivel Ruido = {amp_ruido}", linewidth=1.5)

ax1.set_title("Identificación del Sistema H1(ω) - Entrada: Tonos Puros Provistos", fontsize=12, fontweight='bold')
ax1.set_ylabel("Magnitud Estimada |H1| [dB]", fontsize=10)
ax1.grid(True, which='both', linestyle='--', alpha=0.5)
ax1.set_ylim(-80, 5)
ax1.legend(loc='lower left')

ax2.set_title("Coherencia Cuadrática γ²_xy(ω) - Tonos Puros", fontsize=12, fontweight='bold')
ax2.set_xlabel("Frecuencia [Hz]", fontsize=10)
ax2.set_ylabel("Coherencia γ²", fontsize=10)
ax2.grid(True, which='both', linestyle='--', alpha=0.5)
ax2.set_ylim(-0.05, 1.05)
ax2.legend(loc='lower left')

plt.tight_layout()
plt.show()

# Gráfico 2: Identificación y Coherencia para Entrada Musical
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

for amp_ruido in niveles_ruido:
    freqs, H, coh = resultados_musica[amp_ruido]
    mag_db = 20 * np.log10(np.clip(np.abs(H), 1e-12, None))
    ax1.semilogx(freqs, mag_db, label=f"Nivel Ruido = {amp_ruido}", linewidth=1.5)
    ax2.semilogx(freqs, coh, label=f"Nivel Ruido = {amp_ruido}", linewidth=1.5)

ax1.set_title("Identificación del Sistema H1(ω) - Entrada: Señal Musical Provista", fontsize=12, fontweight='bold')
ax1.set_ylabel("Magnitud Estimada |H1| [dB]", fontsize=10)
ax1.grid(True, which='both', linestyle='--', alpha=0.5)
ax1.set_ylim(-80, 5)
ax1.legend(loc='lower left')

ax2.set_title("Coherencia Cuadrática γ²_xy(ω) - Señal Musical", fontsize=12, fontweight='bold')
ax2.set_xlabel("Frecuencia [Hz]", fontsize=10)
ax2.set_ylabel("Coherencia γ²", fontsize=10)
ax2.grid(True, which='both', linestyle='--', alpha=0.5)
ax2.set_ylim(-0.05, 1.05)
ax2.legend(loc='lower left')

plt.tight_layout()
plt.show()

A partir de la estimación $H_1(\omega)$, se identifica un **filtro paso bajos FIR de fase lineal** (diseñado con ventana Hamming) que presenta una frecuencia de corte a $1000\text{ Hz}$ y una atenucación superior a $-40\text{ dB}$ en la banda de rechazo.

### 9.1 Análisis de la Coherencia por Zonas Espectrales
- **Banda de paso ($0 \le f \le 1000\text{ Hz}$):**  La coherencia $\gamma_{xy}^2(\omega)$ se mantiene alta (próxima a **1** para ruidos bajos de $0.01$). En esta zona la energía de la señal de entrada $G_{xx}(\omega)$ se transmite eficazmente a la salida y domina sobre el ruido de medición, confirmando un comportamiento fuertemente lineal.
- **Banda de rechazo ($f > 1000\text{ Hz}$):**  La coherencia colapsa a valores cercanos a **0**. Dado que el sistema atenuó drásticamente la entrada, la señal capturada a la salida consiste en ruido no correlacionado $n_y[n]$, por lo cual $G_{xy}(\omega) \approx 0$.

### 9.2 Influencia del Nivel de Ruido (SNR)
Al incrementar el nivel de ruido de $0.01$ a $0.05$ y $0.20$, la coherencia en la banda de paso cae sensiblemente. Esto demuestra experimentalmente que el ruido aditivo no correlacionado disminuye el valor de coherencia cuadrática y aumenta el error de la estimación $H_1(\omega)$.

### 9.3 Sistemas Físicos Reales con Comportamientos Análogos
En sistemas físicos reales (ej. electroacústica e ingeniería de sonido):
1. **Altavoces / Bafles:** Muestran alta coherencia en su banda pasante de trabajo, pero colapso de coherencia en frecuencias subbajas (debido a baja eficiencia del cono) o muy altas (por ruptura del diafragma y ruido térmico).
2. **Amplificadores con Saturación (Clipping):** Cuando la amplitud de entrada sobrepasa el rango dinámico del amplificador, aparecen armónicos no presentes en la entrada, generando no linealidades que reducen la coherencia.
3. **Acústica de Salas / Recintos:** El ruido ambiente no correlacionado y las reflexiones difusas reducen la coherencia en los nulos espectrales causados por interferencias destructivas de fase.